# Twitch Channel Analytics — Capstone Project
### Engagement Efficiency Segmentation & Growth Prediction

**DATA110 Capstone (Individual Project)**

This notebook runs the full pipeline: data cleaning → feature engineering → Basic level → Intermediate level → Advanced level (novel contribution).

**Before running:** run the next cell and upload `twitchdata-update.csv` (or `twitch_raw.csv`) when prompted.

In [ ]:
!pip install -q scikit-learn matplotlib pandas openpyxl

import os
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('reports', exist_ok=True)

from google.colab import files
print("Upload your Twitch CSV file:")
uploaded = files.upload()
fname = list(uploaded.keys())[0]
os.rename(fname, 'data/raw/twitch_raw.csv')
print("Saved to data/raw/twitch_raw.csv")

## Step 1 — Data Cleaning
Load the raw dataset, standardize column names, check data quality, save a cleaned version.

In [ ]:
import pandas as pd

RAW_PATH = "data/raw/twitch_raw.csv"
OUT_PATH = "data/processed/twitch_clean.csv"


def load_raw(path: str = RAW_PATH) -> pd.DataFrame:
    df = pd.read_csv(path)
    return df


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename_map = {
        "Channel": "channel",
        "Watch time(Minutes)": "watch_time_min",
        "Stream time(minutes)": "stream_time_min",
        "Peak viewers": "peak_viewers",
        "Average viewers": "avg_viewers",
        "Followers": "followers",
        "Followers gained": "followers_gained",
        "Views gained": "views_gained",
        "Partnered": "partnered",
        "Mature": "mature",
        "Language": "language",
    }
    df = df.rename(columns=rename_map)
    return df


def quality_report(df: pd.DataFrame) -> None:
    print("Shape:", df.shape)
    print("\nMissing values per column:\n", df.isnull().sum())
    print("\nDuplicate channel names:", df["channel"].duplicated().sum())
    print("\nNegative / zero checks (should all be False):")
    numeric_cols = [
        "watch_time_min", "stream_time_min", "peak_viewers",
        "avg_viewers", "followers", "views_gained",
    ]
    for col in numeric_cols:
        print(f"  {col} has values <= 0:", (df[col] <= 0).any())


def clean(df: pd.DataFrame) -> pd.DataFrame:
    df = standardize_columns(df)

    # Drop exact duplicate rows if any slipped in
    before = len(df)
    df = df.drop_duplicates(subset="channel", keep="first")
    after = len(df)
    if before != after:
        print(f"Dropped {before - after} duplicate channel rows.")

    # followers_gained can legitimately be negative (net loss over the
    # tracked period) -> keep as is, this is real signal, not an error.

    # Basic sanity filter: stream_time_min must be > 0 to compute
    # per-minute ratios later without divide-by-zero issues.
    df = df[df["stream_time_min"] > 0].reset_index(drop=True)

    return df

raw = load_raw()
quality_report(standardize_columns(raw.copy()))
cleaned = clean(raw)
cleaned.to_csv(OUT_PATH, index=False)
print(f"\nSaved cleaned data -> {OUT_PATH}  (rows: {len(cleaned)})")


## Step 2 — Feature Engineering
Build the engagement-ratio features and the composite **Engagement Efficiency Score (EES)**.

Ratios are winsorized (capped at 1st/99th percentile) before scaling so a handful of extreme outlier channels don't distort everything else.

In [ ]:
import pandas as pd

IN_PATH = "data/processed/twitch_clean.csv"
OUT_PATH = "data/processed/twitch_features.csv"


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["avg_to_peak_ratio"] = df["avg_viewers"] / df["peak_viewers"]
    df["watch_per_follower"] = df["watch_time_min"] / df["followers"]
    df["views_per_follower"] = df["views_gained"] / df["followers"]
    df["follower_growth_rate"] = df["followers_gained"] / df["followers"]
    df["viewers_per_stream_hr"] = df["avg_viewers"] / (df["stream_time_min"] / 60)
    df["stickiness"] = df["watch_time_min"] / (
        df["stream_time_min"] * df["avg_viewers"]
    )

    return df


def winsorize(series: pd.Series, lower_pct: float = 0.01, upper_pct: float = 0.99) -> pd.Series:
    """Cap extreme outliers at the given percentiles before scaling.

    A handful of channels have very few followers relative to their
    watch time / views, which produces ratio values 5-40x larger than
    the rest of the dataset. Left uncapped, these single points dominate
    min-max scaling and any distance-based clustering on top of it.
    Winsorizing keeps the ranking largely intact while preventing a
    few outliers from compressing everyone else's scaled range.
    """
    lower = series.quantile(lower_pct)
    upper = series.quantile(upper_pct)
    return series.clip(lower=lower, upper=upper)


def min_max_scale(series: pd.Series) -> pd.Series:
    return (series - series.min()) / (series.max() - series.min())


def compute_engagement_efficiency_score(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    ratio_cols = [
        "avg_to_peak_ratio",
        "watch_per_follower",
        "views_per_follower",
        "follower_growth_rate",
        "viewers_per_stream_hr",
        "stickiness",
    ]

    scaled_cols = []
    for col in ratio_cols:
        winsorized = winsorize(df[col])
        scaled_col = f"{col}_scaled"
        df[scaled_col] = min_max_scale(winsorized)
        scaled_cols.append(scaled_col)

    # Equal-weighted composite score (justified in report: no prior basis
    # to weight one engagement dimension over another, so equal weighting
    # is the defensible default; sensitivity to weighting is discussed
    # in Results & Discussion).
    df["engagement_efficiency_score"] = df[scaled_cols].mean(axis=1)

    return df

df = pd.read_csv(IN_PATH)
df = engineer_features(df)
df = compute_engagement_efficiency_score(df)

print("Engineered columns added:")
new_cols = [c for c in df.columns if c not in pd.read_csv(IN_PATH).columns]
print(new_cols)
print("\nEES summary:")
print(df["engagement_efficiency_score"].describe())

df.to_csv(OUT_PATH, index=False)
print(f"\nSaved feature-engineered data -> {OUT_PATH}")


## Step 3 — Basic Level
Exploratory data analysis + a single-feature linear regression (Watch Time → Average Viewers).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

DATA_PATH = "data/processed/twitch_clean.csv"


def run_eda(df: pd.DataFrame) -> None:
    print("=== Basic Summary Statistics ===")
    print(df[["watch_time_min", "avg_viewers", "followers", "peak_viewers"]].describe())

    print("\n=== Top 5 Languages by Channel Count ===")
    print(df["language"].value_counts().head())

    print("\n=== Partnered vs Non-Partnered counts ===")
    print(df["partnered"].value_counts())

    corr = df[["watch_time_min", "stream_time_min", "peak_viewers",
               "avg_viewers", "followers", "followers_gained",
               "views_gained"]].corr()
    print("\n=== Correlation matrix ===")
    print(corr.round(2))


def plot_basic_charts(df: pd.DataFrame) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].scatter(df["watch_time_min"], df["avg_viewers"], alpha=0.5, s=15)
    axes[0].set_xlabel("Watch Time (minutes)")
    axes[0].set_ylabel("Average Viewers")
    axes[0].set_title("Watch Time vs Average Viewers")

    df["language"].value_counts().head(8).plot(kind="bar", ax=axes[1])
    axes[1].set_title("Top 8 Languages by Channel Count")
    axes[1].set_ylabel("Number of Channels")

    plt.tight_layout()
    plt.savefig("reports/basic_eda_charts.png", dpi=150)
    print("Saved chart -> reports/basic_eda_charts.png")


def simple_regression(df: pd.DataFrame) -> None:
    X = df[["watch_time_min"]]
    y = df["avg_viewers"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = LinearRegression()
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    print("\n=== Simple Linear Regression: Watch Time -> Average Viewers ===")
    print(f"Coefficient: {model.coef_[0]:.6f}")
    print(f"Intercept: {model.intercept_:.2f}")
    print(f"R^2 on test set: {r2_score(y_test, preds):.3f}")
    print(f"MAE on test set: {mean_absolute_error(y_test, preds):.2f}")

df = pd.read_csv(DATA_PATH)
run_eda(df)
plot_basic_charts(df)
simple_regression(df)


In [ ]:
from IPython.display import Image
Image('reports/basic_eda_charts.png')

## Step 4 — Intermediate Level
Multi-feature Logistic Regression predicting the `Mature` flag.

**Note:** `Partnered` was considered first but rejected as a target — it's 978/22, so a classifier would hit ~98% accuracy just by always predicting True, without learning anything. `Mature` (770/230) is a much fairer, more meaningful classification problem. This was a deliberate, data-driven decision.

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)

DATA_PATH = "data/processed/twitch_clean.csv"

FEATURES = [
    "watch_time_min", "stream_time_min", "peak_viewers",
    "avg_viewers", "followers", "followers_gained", "views_gained",
]
TARGET = "mature"


def prepare_data(df: pd.DataFrame):
    X = df[FEATURES]
    y = df[TARGET].astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return X_train_scaled, X_test_scaled, y_train, y_test, scaler


def train_and_evaluate(X_train, X_test, y_train, y_test):
    model = LogisticRegression(max_iter=1000, class_weight="balanced")
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    print("=== Logistic Regression: Predicting Mature Flag ===")
    print(f"Accuracy:  {accuracy_score(y_test, preds):.3f}")
    print(f"Precision: {precision_score(y_test, preds):.3f}")
    print(f"Recall:    {recall_score(y_test, preds):.3f}")
    print(f"F1 score:  {f1_score(y_test, preds):.3f}")
    print("\nConfusion matrix:")
    print(confusion_matrix(y_test, preds))
    print("\nFull classification report:")
    print(classification_report(y_test, preds))

    print("Feature coefficients (standardized -> comparable magnitudes):")
    for feat, coef in zip(FEATURES, model.coef_[0]):
        print(f"  {feat:20s} {coef:+.3f}")

    return model

df = pd.read_csv(DATA_PATH)
X_train, X_test, y_train, y_test, scaler = prepare_data(df)
train_and_evaluate(X_train, X_test, y_train, y_test)


## Step 5 — Advanced Level (Primary Research Project)
### Engagement Efficiency Segmentation & Growth Prediction

**Why this is novel:** most public analyses of this dataset predict raw follower counts from raw size features (watch time, peak viewers) — which mostly just re-learns "bigger channels have bigger numbers." Here we engineer six **size-independent engagement ratios** into a composite Engagement Efficiency Score, then:
- **Segment** channels via K-Means clustering
- **Predict** Followers Gained using only the engineered features, comparing 3 models
- **Interpret** which engagement behaviors actually drive growth

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

DATA_PATH = "data/processed/twitch_features.csv"

ENGINEERED_FEATURES = [
    "avg_to_peak_ratio", "watch_per_follower", "views_per_follower",
    "follower_growth_rate", "viewers_per_stream_hr", "stickiness",
]
# Pre-scaled (winsorized + min-max scaled) versions from feature
# engineering step -- used for clustering so extreme outliers don't
# dominate the distance metric.
CLUSTER_FEATURES = [f"{c}_scaled" for c in ENGINEERED_FEATURES]
TARGET = "followers_gained"


# ---------- PART A: CLUSTERING ----------

def find_best_k(X_scaled, k_range=range(2, 7)):
    scores = {}
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X_scaled)
        score = silhouette_score(X_scaled, labels)
        scores[k] = score
        print(f"  k={k}: silhouette score = {score:.3f}")
    best_k = max(scores, key=scores.get)
    print(f"Best k by silhouette score: {best_k}")
    return best_k


def run_clustering(df: pd.DataFrame):
    # Features are already winsorized + min-max scaled in feature
    # engineering (see CLUSTER_FEATURES); re-applying StandardScaler on
    # top just standardizes them to comparable variance for K-Means.
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df[CLUSTER_FEATURES])

    print("=== Selecting number of clusters ===")
    best_k = find_best_k(X_scaled)

    km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
    df["cluster"] = km.fit_predict(X_scaled)

    print(f"\n=== Cluster profile (mean engineered features, k={best_k}) ===")
    profile = df.groupby("cluster")[ENGINEERED_FEATURES + ["engagement_efficiency_score"]].mean()
    print(profile.round(3))

    print("\n=== Cluster sizes ===")
    print(df["cluster"].value_counts().sort_index())

    # Simple 2D visualization using top 2 features by variance for interpretability
    fig, ax = plt.subplots(figsize=(7, 6))
    scatter = ax.scatter(
        df["watch_per_follower"], df["follower_growth_rate"],
        c=df["cluster"], cmap="viridis", alpha=0.6, s=20
    )
    ax.set_xlabel("Watch Time per Follower")
    ax.set_ylabel("Follower Growth Rate")
    ax.set_title(f"Channel Clusters (k={best_k}) by Engagement Pattern")
    plt.colorbar(scatter, label="Cluster")
    plt.tight_layout()
    plt.savefig("reports/advanced_clusters.png", dpi=150)
    print("Saved chart -> reports/advanced_clusters.png")

    return df, best_k


# ---------- PART B: SUPERVISED MODEL COMPARISON ----------

def compare_models(df: pd.DataFrame):
    X = df[ENGINEERED_FEATURES]
    y = df[TARGET]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    models = {
        "Linear Regression": LinearRegression(),
        "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
        "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    }

    results = []
    fitted = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        r2 = r2_score(y_test, preds)
        mae = mean_absolute_error(y_test, preds)
        results.append({"model": name, "r2": r2, "mae": mae})
        fitted[name] = model
        print(f"{name:20s}  R^2={r2:.3f}  MAE={mae:,.0f}")

    results_df = pd.DataFrame(results).sort_values("r2", ascending=False)
    print("\n=== Model comparison (sorted by R^2) ===")
    print(results_df.to_string(index=False))

    best_model_name = results_df.iloc[0]["model"]
    best_model = fitted[best_model_name]
    print(f"\nBest model: {best_model_name}")

    if hasattr(best_model, "feature_importances_"):
        importances = pd.Series(
            best_model.feature_importances_, index=ENGINEERED_FEATURES
        ).sort_values(ascending=False)
        print("\nFeature importances (best model):")
        print(importances.round(3))

        fig, ax = plt.subplots(figsize=(7, 5))
        importances.plot(kind="barh", ax=ax)
        ax.set_title(f"Feature Importance - {best_model_name}")
        ax.invert_yaxis()
        plt.tight_layout()
        plt.savefig("reports/advanced_feature_importance.png", dpi=150)
        print("Saved chart -> reports/advanced_feature_importance.png")

    return results_df, fitted

df = pd.read_csv(DATA_PATH)

print("########## PART A: CLUSTERING ##########\n")
df, best_k = run_clustering(df)
df.to_csv("data/processed/twitch_clustered.csv", index=False)
print("Saved -> data/processed/twitch_clustered.csv")

print("\n########## PART B: MODEL COMPARISON ##########\n")
results_df, fitted_models = compare_models(df)
results_df.to_csv("reports/model_comparison_results.csv", index=False)
print("Saved -> reports/model_comparison_results.csv")


In [ ]:
from IPython.display import Image
Image('reports/advanced_clusters.png')

In [ ]:
from IPython.display import Image
Image('reports/advanced_feature_importance.png')

## Summary of Results

| Level | Task | Result |
|---|---|---|
| Basic | Linear Regression (Watch Time → Avg Viewers) | R² = 0.406 |
| Intermediate | Logistic Regression (predict `Mature`) | Accuracy = 0.475, F1 = 0.348 |
| Advanced | K-Means clustering | k=2, silhouette = 0.386 |
| Advanced | Gradient Boosting (predict `followers_gained`) | R² = 0.337 (best of 3 models) |

**Key insight:** High-Engagement-Efficiency channels (niche, high watch-time-per-follower) do *not* gain the most absolute followers — Cluster 0 (high EES) averages ~81K followers gained vs. Cluster 1 (standard) at ~220K. Raw follower growth is still partly a function of channel size, not efficiency alone. This is the core finding of the advanced-level analysis.